# 02_strategy_generator.ipynb - Generador de Estrategias

**PENDIENTE DE IMPLEMENTAR** - Este notebook será el motor que genera estrategias.

## Flujo del proyecto
1. 01_eda.ipynb → EDA + datos
2. batch_calibrate.py → Csl + BP calibrados → `data/universe_admitted.csv`
3. **02_strategy_generator.ipynb ← ESTE: Genera estrategias**
4. 03_backtest_strategies.ipynb → Backtestea estrategias generadas
5. 04_robustez.ipynb → Tests robustez (Sección 9)
6. 05_validacion.ipynb → Validación final (Sección 10)

## Qué debe hacer este generador (según metodología Sección 3, 7-8)

### Opciones de implementación

| Enfoque | Descripción | Complejidad |
|---------|-------------|-------------|
| **Rule-Based Builder + Random Search** | Combinar bloques predefinidos (filtros, triggers, exits), probar N combinaciones aleatorias, keep top-K | Media-Baja |
| **Genético (DEAP)** | Población de reglas, cruce/mutación, fitness=PnL/DD | Media |
| **Genetic Programming (GP)** | Árboles de expresión, evoluciona lógica completa | Alta |
| **Grid Search sistemático** | Probar todas las combinaciones en rangos definidos | Baja (pero explosión combinatoria) |

### Recomendación: Rule-Based Builder + Random Search

1. **Definir bloques parametrizables**:
   - **Filtros**: ER, RSI, MACD, ADX, BB, Volumen, Gaps, Hora/Día
   - **Triggers**: Cruce SMA, Ruptura Donchian, Pullback a media, Patrón vela (engulfing, hammer, etc.)
   - **Exits**: TP/SL fijos (ratio Ctp/Csl), Trailing ATR, Time-based, Indicador inverso

2. **Random Search**: Generar N combinaciones aleatorias → Backtest rápido (un activo o muestra) → Keep top-K por fitness

3. **Refinamiento**: Top-K → Búsqueda local / Walk-forward en universo completo

4. **Exportar**: Reglas finales en formato serializable (JSON/YAML) para backtest completo

## Estructura de una Estrategia (formato de salida)

In [ ]:
# Ejemplo de estructura de estrategia generada
strategy_template = {
    "name": "strat_001",
    "description": "ER filter + SMA trend + Donchian breakout",
    "version": "1.0",
    "created_at": "2026-06-25T13:00:00",
    "filters": [
        {"type": "efficiency_ratio", "params": {"k": 10, "threshold": 0.3, "direction": "long"}},
        {"type": "sma_trend", "params": {"period": 10, "direction": "above"}},
        {"type": "volume_confirm", "params": {"ma_period": 20, "multiplier": 1.2}}
    ],
    "trigger": {
        "type": "donchian_breakout",
        "params": {"lookback": 20, "breakout_type": "high"}
    },
    "exit": {
        "type": "atr_tp_sl",
        "params": {"csl": "AUTO", "tp_ratio": 1.5, "max_bars": 40}
    },
    "universe": "admitted",  # usa data/universe_admitted.csv
    "fitness": {
        "function": "pnl_over_dd",
        "value": 0.0,
        "metrics": {"sharpe": 0.0, "win_rate": 0.0, "profit_factor": 0.0}
    }
}

import json
print(json.dumps(strategy_template, indent=2))

## Próximos pasos para implementar

1. **Crear `src/strategy_builder.py`** con clases:
   - `FilterBlock`, `TriggerBlock`, `ExitBlock`
   - `Strategy` (composición de bloques)
   - `StrategyGenerator` (random search, evaluación, selección)
   - `FitnessEvaluator` (PnL/DD, Sharpe, etc.)

2. **Definir bloques base** en `src/strategy_blocks.py`

3. **Implementar random search** en este notebook

4. **Exportar estrategias** a `data/strategies_generated.json`

---
**NOTA**: El backtester genérico ya existe en `src/risk.py` (`run_backtest_loop`, `calculate_performance_metrics`). El generador solo necesita construir la señal de entrada booleana y pasársela al backtester.